# 07 — MITRE Tactic Multi-Label Classification on TRAM

**Objective:** fine-tune BERT to predict which MITRE ATT&CK tactics a sentence describes, using the TRAM data we prepared in notebook 02 and the multi-label mechanics we established in notebook 06.

## The task in one line

Given a sentence like *"The attacker phished credentials and moved laterally through the network"*, output the set of MITRE tactics it describes, e.g. `{initial-access, lateral-movement}`.

## Why this is the payoff notebook

- Notebook 02 gave us real labeled data — sentences tagged with their tactics.
- Notebook 06 gave us the mechanics — sigmoid heads, BCE loss, macro F1, threshold tuning.
- This notebook combines both and actually trains a useful model.

## How the multi-label switch lands in HuggingFace

`BertForSequenceClassification` is the same class we used for single-label in CTI 101 notebook 08. The **only** thing that changes to make it multi-label:

```python
model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-cased',
    num_labels=14,
    problem_type='multi_label_classification',   # <-- this line
)
```

That flag tells HuggingFace to use `BCEWithLogitsLoss` instead of `CrossEntropyLoss`. Labels must then be **float multi-hot vectors** (which is exactly the format notebook 02 saved).

## What this notebook does

1. Load the prepared TRAM tactics dataset from `processed/tram_tactics/`
2. Tokenize sentences
3. Configure BERT for multi-label classification
4. Train with macro-F1 as the selection metric
5. Tune per-tactic decision thresholds on validation
6. Evaluate on test with tuned thresholds — per-tactic F1 + headline metrics
7. Save the trained model to `models/tactic_clf_final/` for notebooks 08, 09, 10

## Prerequisite

Notebook 02 has been run; `processed/tram_tactics/` exists.

## Step 1 — Load the prepared dataset

In [1]:
import json
from pathlib import Path
from datasets import load_from_disk

DATA_PATH = Path('processed/tram_tactics')
assert DATA_PATH.exists(), f'Run notebook 02 first — {DATA_PATH} does not exist.'

ds = load_from_disk(str(DATA_PATH))
print(ds)

with open(DATA_PATH / 'tactic_vocab.json') as f:
    vocab = json.load(f)
tactics = vocab['tactics']
tactic2id = vocab['tactic2id']
id2tactic = {i: t for t, i in tactic2id.items()}
num_labels = len(tactics)

print(f'\n{num_labels} tactics:')
for t in tactics:
    print(f'  {t}')

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'tactic_names'],
        num_rows: 3256
    })
    validation: Dataset({
        features: ['text', 'labels', 'tactic_names'],
        num_rows: 407
    })
    test: Dataset({
        features: ['text', 'labels', 'tactic_names'],
        num_rows: 407
    })
})

14 tactics:
  collection
  command-and-control
  credential-access
  defense-evasion
  discovery
  execution
  exfiltration
  impact
  initial-access
  lateral-movement
  persistence
  privilege-escalation
  reconnaissance
  resource-development


### Look at one example

Sanity-check the label format — the key thing is that `labels` is a list of 14 floats (multi-hot), not a single integer.

In [2]:
ex = ds['train'][0]
print(f'text:         {ex["text"][:150]}...')
print(f'tactic_names: {ex["tactic_names"]}')
print(f'labels:       {ex["labels"]}')
print(f'labels shape: {len(ex["labels"])} floats (one per tactic)')

text:         Some of these nodes operate as part of an encrypted proxy service to prevent attribution by concealing their country of origin and TTPs. ...
tactic_names: ['command-and-control']
labels:       [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
labels shape: 14 floats (one per tactic)


## Step 2 — Tokenize

Same tokenizer recipe as CTI 101 notebook 08. TRAM sentences are short (mostly under 128 subwords), so 128 is plenty — lets us use larger batches and train faster than with 256.

In [3]:
from transformers import AutoTokenizer

MODEL_CKPT = 'bert-base-cased'
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)


def tokenize(batch):
    # 'labels' and 'tactic_names' pass through untouched.
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LEN)


tokenized = ds.map(tokenize, batched=True)
# Drop columns the model doesn't consume; keep 'labels' (required).
cols_to_drop = ['text', 'tactic_names']
tokenized = tokenized.remove_columns([c for c in cols_to_drop if c in tokenized['train'].column_names])
print(tokenized)

Map:   0%|          | 0/3256 [00:00<?, ? examples/s]

Map:   0%|          | 0/407 [00:00<?, ? examples/s]

Map:   0%|          | 0/407 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3256
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 407
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 407
    })
})


## Step 3 — Configure BERT for multi-label

The one-line switch that turns this into a multi-label model:

```python
problem_type='multi_label_classification'
```

In [4]:
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=num_labels,
    problem_type='multi_label_classification',
    id2label=id2tactic,
    label2id=tactic2id,
)

collator = DataCollatorWithPadding(tokenizer)

print(f'Base model: {MODEL_CKPT}')
print(f'Classification head: {num_labels} sigmoid outputs (one per tactic)')

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Base model: bert-base-cased
Classification head: 14 sigmoid outputs (one per tactic)


## Step 4 — Metrics

The multi-label metric set from notebook 06: subset accuracy, Hamming loss, micro-F1, macro-F1. Macro F1 is what we'll use to pick the best checkpoint — it refuses to let common tactics dominate the score.

At training/validation time we use the default 0.5 threshold. Per-tactic threshold tuning happens *after* training on the final model, against the validation set.

In [5]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    hamming_loss,
    f1_score,
    precision_recall_fscore_support,
)


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = sigmoid(logits)
    preds = (probs >= 0.5).astype(np.float32)
    return {
        'subset_accuracy': accuracy_score(labels, preds),
        'hamming_accuracy': 1.0 - hamming_loss(labels, preds),
        'micro_f1': f1_score(labels, preds, average='micro', zero_division=0),
        'macro_f1': f1_score(labels, preds, average='macro', zero_division=0),
    }

## Step 5 — Train

TRAM is small (a few thousand sentences), so training is fast — maybe 15–20 min on CPU, a couple of minutes on GPU. We train for more epochs than CTI 101 did on its toy dataset (8) because real multi-label tasks tend to plateau later.

In [6]:
from transformers import TrainingArguments, Trainer

OUT_DIR = Path('models/tactic_clf')
OUT_DIR.parent.mkdir(exist_ok=True)

args = TrainingArguments(
    output_dir=str(OUT_DIR),
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print('\nFinal training metrics:')
for k, v in train_result.metrics.items():
    print(f'  {k}: {v}')

/tmp/ipykernel_258541/1254454757.py:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Subset Accuracy,Hamming Accuracy,Micro F1,Macro F1
1,0.280400,0.270511,0.179361,0.901895,0.314110,0.047250
2,0.210800,0.206033,0.324324,0.924008,0.501726,0.177003
3,0.163300,0.168923,0.466830,0.942260,0.657648,0.379792
4,0.122500,0.150613,0.557740,0.949280,0.728638,0.439332
5,0.094700,0.137954,0.619165,0.956827,0.772643,0.528449
6,0.085900,0.132607,0.619165,0.957529,0.778793,0.537496
7,0.075300,0.133572,0.626536,0.957002,0.777475,0.536826
8,0.063100,0.131577,0.621622,0.956651,0.774429,0.545900



Final training metrics:
  train_runtime: 376.878
  train_samples_per_second: 69.115
  train_steps_per_second: 4.33
  total_flos: 1441321414863936.0
  train_loss: 0.1601051260151115
  epoch: 8.0


### What to watch in the training log

- **Loss** should decrease smoothly. BCE losses settle at small absolute values (0.1–0.3 range) because most labels are negative — that's expected, don't read too much into it.
- **`eval_macro_f1`** is the number to watch. If it plateaus early and never climbs much, the model is underfitting — increase epochs or learning rate. If it peaks and then falls, `load_best_model_at_end` already saved the peak checkpoint.
- **Gap between micro and macro F1**: a big gap means common tactics are carrying the score. Threshold tuning (next step) usually helps the macro side more.

## Step 6 — Per-tactic threshold tuning on validation

We now take the best-checkpoint model and do what notebook 06 demonstrated on toy data: for each of the 14 tactics, sweep thresholds on the validation set and pick the one that maximizes that tactic's F1.

This is a cheap post-training step — it just replays predictions over a few threshold values.

In [7]:
# Get raw logits on validation
val_out = trainer.predict(tokenized['validation'])
val_probs = sigmoid(val_out.predictions)
val_labels = val_out.label_ids

thresholds_to_try = np.linspace(0.05, 0.95, 37)
best_thresholds = np.full(num_labels, 0.5)

print(f'{"tactic":28s}  {"best_t":>8s}  {"F1@best":>8s}  {"F1@0.5":>8s}  gain')
print('-' * 68)
for i, name in enumerate(tactics):
    f1s = [f1_score(val_labels[:, i], (val_probs[:, i] >= t).astype(np.float32),
                    zero_division=0) for t in thresholds_to_try]
    best_idx = int(np.argmax(f1s))
    best_thresholds[i] = thresholds_to_try[best_idx]
    f1_default = f1_score(val_labels[:, i],
                          (val_probs[:, i] >= 0.5).astype(np.float32),
                          zero_division=0)
    print(f'  {name:26s}  {best_thresholds[i]:8.2f}  {f1s[best_idx]:8.4f}  {f1_default:8.4f}  {f1s[best_idx] - f1_default:+.4f}')

tactic                          best_t   F1@best    F1@0.5  gain
--------------------------------------------------------------------
  collection                      0.30    0.6667    0.6400  +0.0267
  command-and-control             0.42    0.6963    0.6767  +0.0196
  credential-access               0.50    0.8846    0.8846  +0.0000
  defense-evasion                 0.30    0.8571    0.8397  +0.0175
  discovery                       0.45    0.7089    0.6842  +0.0247
  execution                       0.42    0.8161    0.7683  +0.0478
  exfiltration                    0.12    0.7143    0.0000  +0.7143
  impact                          0.05    0.0000    0.0000  +0.0000
  initial-access                  0.17    0.8525    0.7451  +0.1074
  lateral-movement                0.50    0.8485    0.8485  +0.0000
  persistence                     0.20    0.8039    0.7556  +0.0484
  privilege-escalation            0.57    0.8205    0.8000  +0.0205
  reconnaissance                  0.05    0.0000  

**Reading this:** tactics where "best_t" is much lower than 0.5 are rare classes — the model's probabilities for them never cross 0.5, so default threshold gives F1=0. Lowering the threshold to where probabilities actually concentrate recovers useful F1.

Don't over-trust per-tactic thresholds for tactics with very small validation support (say, <20 positive examples) — the chosen threshold is noisy. You may want to floor at 0.3 for those, or use a global threshold tuned on pooled probabilities.

## Step 7 — Evaluate on test with tuned thresholds

In [8]:
test_out = trainer.predict(tokenized['test'])
test_probs = sigmoid(test_out.predictions)
test_labels = test_out.label_ids

preds_default = (test_probs >= 0.5).astype(np.float32)
preds_tuned = (test_probs >= best_thresholds).astype(np.float32)


def all_metrics(y, yhat):
    return {
        'subset_acc': accuracy_score(y, yhat),
        'hamming_acc': 1.0 - hamming_loss(y, yhat),
        'micro_f1': f1_score(y, yhat, average='micro', zero_division=0),
        'macro_f1': f1_score(y, yhat, average='macro', zero_division=0),
    }


m_def = all_metrics(test_labels, preds_default)
m_tun = all_metrics(test_labels, preds_tuned)

print(f'{"metric":14s}  {"@0.5":>10s}  {"@tuned":>10s}  gain')
print('-' * 48)
for k in m_def:
    print(f'{k:14s}  {m_def[k]:10.4f}  {m_tun[k]:10.4f}  {m_tun[k] - m_def[k]:+.4f}')

print('\nPer-tactic F1 on test (with tuned thresholds):')
per = precision_recall_fscore_support(test_labels, preds_tuned, average=None, zero_division=0)
for i, name in enumerate(tactics):
    p, r, f, sup = per[0][i], per[1][i], per[2][i], per[3][i]
    print(f'  {name:26s}  P={p:.3f}  R={r:.3f}  F1={f:.3f}  support={sup}')

metric                @0.5      @tuned  gain
------------------------------------------------
subset_acc          0.6314      0.6192  -0.0123
hamming_acc         0.9556      0.9528  -0.0028
micro_f1            0.7755      0.7749  -0.0006
macro_f1            0.5231      0.5660  +0.0429

Per-tactic F1 on test (with tuned thresholds):
  collection                  P=0.750  R=0.500  F1=0.600  support=18
  command-and-control         P=0.750  R=0.729  F1=0.739  support=70
  credential-access           P=0.933  R=0.667  F1=0.778  support=21
  defense-evasion             P=0.863  R=0.867  F1=0.865  support=196
  discovery                   P=0.700  R=0.618  F1=0.656  support=34
  execution                   P=0.827  R=0.736  F1=0.779  support=91
  exfiltration                P=0.571  R=0.444  F1=0.500  support=9
  impact                      P=0.000  R=0.000  F1=0.000  support=0
  initial-access              P=0.688  R=0.688  F1=0.688  support=32
  lateral-movement            P=0.875  R=0.667

### Interpreting the per-tactic table

- **Zero F1 with zero support** → that tactic didn't appear in the test split at all. Not a model failure, a data issue; rerun notebook 02 with iterative stratified sampling if this bothers you.
- **Zero F1 with nonzero support** → the model never predicts this tactic. Either the threshold is still too high or the training set has too few examples for the model to learn its signal.
- **High F1 for the dominant tactics** → expected; they have the most training data.
- **Tactics with high precision but low recall** → the model predicts them rarely but accurately. Consider lowering the threshold further.
- **Low precision, high recall** → over-prediction. Raise the threshold.

## Step 8 — Save the final model

We save the model, tokenizer, tactic vocabulary, and tuned thresholds together. Notebooks 08, 09, and 10 load all four as a unit.

In [9]:
FINAL_DIR = Path('models/tactic_clf_final')
FINAL_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))

with open(FINAL_DIR / 'tactic_vocab.json', 'w') as f:
    json.dump({'tactics': tactics, 'tactic2id': tactic2id}, f, indent=2)

with open(FINAL_DIR / 'thresholds.json', 'w') as f:
    json.dump({
        'per_tactic': {tactics[i]: float(best_thresholds[i]) for i in range(num_labels)},
        'default': 0.5,
    }, f, indent=2)

print(f'Saved to {FINAL_DIR}:')
for p in sorted(FINAL_DIR.iterdir()):
    print(f'  {p.name}  ({p.stat().st_size / 1e6:.2f} MB)')

Saved to models/tactic_clf_final:
  config.json  (0.00 MB)
  model.safetensors  (433.31 MB)
  special_tokens_map.json  (0.00 MB)
  tactic_vocab.json  (0.00 MB)
  thresholds.json  (0.00 MB)
  tokenizer.json  (0.67 MB)
  tokenizer_config.json  (0.00 MB)
  training_args.bin  (0.01 MB)
  vocab.txt  (0.21 MB)


## Step 9 — Quick qualitative check

Run the trained model on a hand-written sentence and see which tactics get predicted with which probabilities. This is where the model's behavior becomes tangible.

In [10]:
import torch

device = next(model.parameters()).device
model.eval()

sample_sentences = [
    'The attacker phished credentials and then moved laterally through the network.',
    'The malware exfiltrated sensitive documents over DNS tunneling to a remote server.',
    'A scheduled task was created to maintain persistence across reboots.',
]

for s in sample_sentences:
    enc = tokenizer(s, return_tensors='pt', truncation=True, max_length=MAX_LEN).to(device)
    with torch.no_grad():
        probs = torch.sigmoid(model(**enc).logits[0]).cpu().numpy()

    predicted = [(tactics[i], probs[i]) for i in range(num_labels)
                 if probs[i] >= best_thresholds[i]]
    predicted.sort(key=lambda x: -x[1])

    print(f'\n> {s}')
    if not predicted:
        print('  (no tactics above threshold)')
    for name, p in predicted:
        print(f'  {name:26s}  prob={p:.3f}  threshold={float(best_thresholds[tactic2id[name]]):.2f}')


> The attacker phished credentials and then moved laterally through the network.
  privilege-escalation        prob=0.938  threshold=0.57
  defense-evasion             prob=0.938  threshold=0.30
  persistence                 prob=0.925  threshold=0.20
  initial-access              prob=0.919  threshold=0.17

> The malware exfiltrated sensitive documents over DNS tunneling to a remote server.
  exfiltration                prob=0.133  threshold=0.12

> A scheduled task was created to maintain persistence across reboots.
  privilege-escalation        prob=0.939  threshold=0.57
  persistence                 prob=0.909  threshold=0.20
  execution                   prob=0.828  threshold=0.42


## Summary

| | What |
|---|---|
| Base model | `bert-base-cased` |
| Task | Multi-label sentence classification |
| Data | TRAM II sentences → MITRE tactics (from notebook 02) |
| Classes | 14 MITRE ATT&CK Enterprise tactics |
| Key switch | `problem_type='multi_label_classification'` |
| Headline metric | macro F1 with per-tactic tuned thresholds |
| Saved to | `models/tactic_clf_final/` |

### What to expect, numerically

Macro F1 in the 0.3–0.5 range is a realistic starting point for this dataset. It's not a great number in absolute terms, but:

- Training set is small (~4k sentences, 14 classes → a few hundred per class on average, much less for rare tactics).
- Some tactics are genuinely hard to distinguish from a single sentence alone — a human analyst would often want surrounding context.
- Notebook 08 (Longformer on full reports) and notebook 10 (domain-pretrained SecureBERT) are both levers to push this higher.

**Next:** notebook 08 swaps BERT's 512-token limit for Longformer's 4096-token window and classifies *full reports* instead of sentences.